### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [4]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: BordaRAG- Resolving Knowledge Conflict inRetrieval-Augmented Generation via Borda Voting Process.pdf
  ✓ Loaded 12 pages

Processing: Contradiction Detection in RAG Systems- Evaluating LLMs as Context Validators for Improved Information Consistency.pdf
  ✓ Loaded 13 pages

Processing: KARMA- Leveraging Multi-Agent LLMs for Automated Knowledge Graph Enrichment.pdf
  ✓ Loaded 26 pages

Total documents loaded: 51


In [5]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [6]:
chunks=split_documents(all_pdf_documents)
chunks

Split 51 documents into 234 chunks

Example chunk:
Content: . 
. 
Latest updates: hps://dl.acm.org/doi/10.1145/3746252.3761038
. 
. 
RESEARCH-ARTICLE
BordaRAG: Resolving Knowledge Conflict in Retrieval-Augmented
Generation via Borda Voting Process
YUXIN LI, R...
Metadata: {'producer': 'iText 4.2.0 by 1T3XT', 'creator': 'PyPDF', 'creationdate': '2026-03-07T00:34:23-08:00', 'moddate': '2026-03-07T00:34:23-08:00', 'title': 'BordaRAG: Resolving Knowledge Conflict in Retrieval-Augmented Generation via Borda Voting Process', 'source': '../data/pdf/BordaRAG- Resolving Knowledge Conflict inRetrieval-Augmented Generation via Borda Voting Process.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'BordaRAG- Resolving Knowledge Conflict inRetrieval-Augmented Generation via Borda Voting Process.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'iText 4.2.0 by 1T3XT', 'creator': 'PyPDF', 'creationdate': '2026-03-07T00:34:23-08:00', 'moddate': '2026-03-07T00:34:23-08:00', 'title': 'BordaRAG: Resolving Knowledge Conflict in Retrieval-Augmented Generation via Borda Voting Process', 'source': '../data/pdf/BordaRAG- Resolving Knowledge Conflict inRetrieval-Augmented Generation via Borda Voting Process.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'BordaRAG- Resolving Knowledge Conflict inRetrieval-Augmented Generation via Borda Voting Process.pdf', 'file_type': 'pdf'}, page_content=". \n. \nLatest updates: h\ue03cps://dl.acm.org/doi/10.1145/3746252.3761038\n. \n. \nRESEARCH-ARTICLE\nBordaRAG: Resolving Knowledge Conflict in Retrieval-Augmented\nGeneration via Borda Voting Process\nYUXIN LI, Renmin University of China, Beijing, China\n. \nCHEN XU, Renmin University of China, Beijing, China\n. \nJUN XU, Renmin University of China, Beijing, China\n. \nJI-RONG WEN, Renmin Univers

### Embeddings and vectorStoreDB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

/Users/kaziakibzaoad/ACL-BIONLP/Code/ytrag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17752.76it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [9]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

## initialize the vector store
vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [11]:
# Convert the texts to embeddings
texts = [doc.page_content for doc in chunks]

#generate the embeddings
embeddings=embedding_manager.generate_embeddings(texts)

# Add documents and embeddings to the vector store
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 234 texts...


Batches: 100%|██████████| 8/8 [00:02<00:00,  3.04it/s]

Generated embeddings with shape: (234, 384)
Adding 234 documents to vector store...
Successfully added 234 documents to vector store
Total documents in collection: 234


### Retriever Pipeline from VectorStore

In [12]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [13]:
rag_retriever.retrieve("how contradictions are detected?")

Retrieving documents for query: 'how contradictions are detected?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.41it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_5958b667_116',
  'content': 'its highest accuracy of 0.893 on pair contradictions,\nwhile Claude-3 Sonnet with CoT reaches 0.831 for\nthe same type. Conditional contradictions and self-\ncontradictions prove to be more challenging to de-\ntect, with generally lower accuracy rates across all\nmodels. Self-contradictions show particularly low\ndetection rates, with accuracies ranging from 0.006\nto 0.456, suggesting that identifying contradictions\nwithin a single document is difficult for LLMs. The\nrelative difficulty of contradiction types follows a\nconsistent pattern across models: pair contradic-\ntions are the easiest to detect, followed by condi-\ntional contradictions, while self-contradictions are\ngenerally the most challenging. This hierarchy\nsuggests that LLMs are better equipped to com-\npare and contrast information across distinct\ndocuments than to analyze internal consistency\nwithin a single document or understand complex\nconditional relationships.\nRQ2:

In [27]:
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage

In [28]:
class LocalLLM:
    def __init__(self, model_name: str = "llama3.2"):
        self.model_name = model_name
        
        self.llm = ChatOllama(
            model=self.model_name,
            temperature=0.1,
            num_predict=1024  # replaces max_tokens
        )
        
        print(f"Initialized LOCAL LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str) -> str:
        
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer:"""
        )
        
        formatted_prompt = prompt_template.format(
            context=context, 
            question=query
        )
        
        try:
            response = self.llm.invoke(formatted_prompt)
            return response.content
        except Exception as e:
            return f"Error generating response: {str(e)}"

    def generate_response_simple(self, query: str, context: str) -> str:
        
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            response = self.llm.invoke(simple_prompt)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [32]:
## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    try:
        response = llm.generate_response(query, context)
        return response
    except Exception as e:
        return f"Error generating response: {str(e)}"
    

In [33]:
try:
    local_llm = LocalLLM(model_name="llama3.2")
    print("Local LLM initialized successfully!")
except Exception as e:
    print(f"Failed to initialize local LLM: {e}")
    local_llm = None

Initialized LOCAL LLM with model: llama3.2
Local LLM initialized successfully!


In [34]:
answer = rag_simple(
    "Tell me about Knowledge Graph Construction",
    rag_retriever,
    local_llm
)

print(answer)

Retrieving documents for query: 'Tell me about Knowledge Graph Construction'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.21it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Knowledge Graph Construction refers to the process of transforming unstructured text into a structured representation of entities and their relationships. This involves various techniques such as:

1. Entity Disambiguation: Identifying and resolving ambiguities in entity names.
2. Relationship Extraction: Automatically extracting relationships between entities from text.
3. Schema Alignment: Aligning extracted knowledge with existing domain-specific schemas.

The quest for Knowledge Graph Construction has evolved through three generations of technical paradigms, including:

1. First-generation systems (1990s-2010s): WordNet and other lexical resources.
2. Second-generation systems: Focus on entity disambiguation and relationship extraction.
3. Third-generation systems: Emphasis on multi-agent conversation frameworks, long document understanding, and large-scale biomedical knowledge graphs.

Recent advancements in Knowledge Graph Construction include the development of:

1. Multi-agent 